In [1]:
!pip install -q transformers torch bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.3 MB/s eta 0:00:00


In [6]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Still using CPU")

GPU available: True
GPU: Tesla T4


In [7]:
import os

os.listdir('/content')

['.config',
 'test-balanced.csv.bz2',
 'archive.zip',
 'train-balanced.csv.bz2',
 'sample_data']

In [9]:
import zipfile
import os

with zipfile.ZipFile('/content/archive.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print(os.listdir('/content/dataset'))

['Sarcasm_Headlines_Dataset_v2.json', 'Sarcasm_Headlines_Dataset.json']


In [10]:
import os

print(os.listdir('/content/dataset'))

['Sarcasm_Headlines_Dataset_v2.json', 'Sarcasm_Headlines_Dataset.json']


In [11]:
import json
import pandas as pd

data = []

with open('/content/dataset/Sarcasm_Headlines_Dataset_v2.json', 'r') as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [12]:
print("Sample 1:")
print(df.iloc[0])

print("\nSample 2:")
print(df.iloc[1])

Sample 1:
is_sarcastic                                                    1
headline        thirtysomething scientists unveil doomsday clo...
article_link    https://www.theonion.com/thirtysomething-scien...
Name: 0, dtype: object

Sample 2:
is_sarcastic                                                    0
headline        dem rep. totally nails why congress is falling...
article_link    https://www.huffingtonpost.com/entry/donna-edw...
Name: 1, dtype: object


In [13]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["is_sarcastic"]
)

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

Training samples: 25757
Validation samples: 2862


In [14]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    output_attentions=True
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
from torch.utils.data import Dataset

class SarcasmDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.texts = dataframe["headline"].tolist()
        self.labels = dataframe["is_sarcastic"].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=64,
            return_tensors="pt"
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

In [16]:
train_dataset = SarcasmDataset(train_df, tokenizer)
val_dataset = SarcasmDataset(val_df, tokenizer)

print(train_dataset[0])

{'input_ids': tensor([    0,   102,   218,  5618, 20125,  5662,    74,    28,  2702,    13,
            5,   232,    35,   542,   659,   834,     2,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'labels': tensor(0)}


In [17]:
from transformers import TrainingArguments, Trainer

In [18]:
training_args = TrainingArguments(
    output_dir="./roberta-sarcasm",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=100,
    report_to="none"
)

In [19]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [20]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.222035,0.209215


TrainOutput(global_step=1610, training_loss=0.26188520511485036, metrics={'train_runtime': 314.056, 'train_samples_per_second': 82.014, 'train_steps_per_second': 5.126, 'total_flos': 847118931613440.0, 'train_loss': 0.26188520511485036, 'epoch': 1.0})

In [22]:
model = trainer.model
model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [23]:
text1 = df.iloc[0]["headline"]
text2 = df.iloc[1]["headline"]

print("Sample 1:", text1)
print("Sample 2:", text2)

Sample 1: thirtysomething scientists unveil doomsday clock of hair loss
Sample 2: dem rep. totally nails why congress is falling short on gender, racial equality


In [24]:
inputs1 = tokenizer(
    text1,
    return_tensors="pt",
    truncation=True,
    max_length=64
)

inputs1 = {k: v.to(model.device) for k, v in inputs1.items()}

with torch.no_grad():
    outputs1 = model(**inputs1)

attentions1 = outputs1.attentions

print("Number of layers:", len(attentions1))
print("Layer shape:", attentions1[0].shape)

Number of layers: 12
Layer shape: torch.Size([1, 12, 13, 13])


In [25]:
prediction1 = torch.argmax(outputs1.logits, dim=1).item()

print("Prediction:", prediction1)
print("Actual:", df.iloc[0]["is_sarcastic"])

Prediction: 1
Actual: 1


In [26]:
inputs2 = tokenizer(
    text2,
    return_tensors="pt",
    truncation=True,
    max_length=64
)

inputs2 = {k: v.to(model.device) for k, v in inputs2.items()}

with torch.no_grad():
    outputs2 = model(**inputs2)

attentions2 = outputs2.attentions

prediction2 = torch.argmax(outputs2.logits, dim=1).item()

print("Prediction:", prediction2)
print("Actual:", df.iloc[1]["is_sarcastic"])

Prediction: 0
Actual: 0


In [27]:
attention1 = attentions1[11].detach().cpu()
attention2 = attentions2[11].detach().cpu()

In [28]:
from bertviz import head_view

tokens1 = tokenizer.convert_ids_to_tokens(
    inputs1["input_ids"][0]
)

head_view(
    attention1.unsqueeze(0),
    tokens1
)

<IPython.core.display.Javascript object>

In [29]:
import torch

# Average all attention heads
avg_attention1 = attention1.mean(dim=1).squeeze(0)
avg_attention2 = attention2.mean(dim=1).squeeze(0)

print(avg_attention1.shape)
print(avg_attention2.shape)

torch.Size([13, 13])
torch.Size([17, 17])


In [30]:
tokens1 = tokenizer.convert_ids_to_tokens(inputs1["input_ids"][0])
tokens2 = tokenizer.convert_ids_to_tokens(inputs2["input_ids"][0])

print(tokens1)
print(tokens2)

['<s>', 'th', 'irty', 'something', 'Ġscientists', 'Ġunveil', 'Ġdo', 'omsday', 'Ġclock', 'Ġof', 'Ġhair', 'Ġloss', '</s>']
['<s>', 'dem', 'Ġrep', '.', 'Ġtotally', 'Ġnails', 'Ġwhy', 'Ġcongress', 'Ġis', 'Ġfalling', 'Ġshort', 'Ġon', 'Ġgender', ',', 'Ġracial', 'Ġequality', '</s>']


In [32]:
import re

# Words/tokens we DON'T want
stopwords = {
    "the", "a", "an", "and", "or", "but", "if", "then",
    "is", "are", "was", "were", "be", "been", "being",
    "to", "of", "in", "on", "at", "for", "from", "with",
    "by", "as", "it", "this", "that", "these", "those",
    "he", "she", "they", "we", "you", "i", "his", "her",
    "their", "our", "your", "my", "me", "him", "them",
    "do", "does", "did", "has", "have", "had",
    "will", "would", "could", "should", "can", "may",
    "not", "no"
}

def get_meaningful_top3(tokens, avg_attention):

    # Average attention received by each token
    token_scores = avg_attention.mean(dim=0)

    candidates = []

    for i, token in enumerate(tokens):

        # Remove special tokens
        if token in ["<s>", "</s>", "<pad>", "<unk>"]:
            continue

        # Remove RoBERTa's word-start marker
        clean_token = token.replace("Ġ", "").strip()

        # Remove empty tokens
        if clean_token == "":
            continue

        # Remove punctuation / symbols
        if not re.search(r"[A-Za-z]", clean_token):
            continue

        # Remove very short garbage tokens
        if len(clean_token) <= 2:
            continue

        # Remove common stopwords
        if clean_token.lower() in stopwords:
            continue

        # Remove tokens containing weird special characters
        if not re.match(r"^[A-Za-z][A-Za-z'-]*$", clean_token):
            continue

        candidates.append(
            (clean_token, float(token_scores[i]))
        )

    # Sort by attention score
    candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Keep top 3 meaningful tokens
    return candidates[:3]


top3_1 = get_meaningful_top3(
    tokens1,
    avg_attention1
)

top3_2 = get_meaningful_top3(
    tokens2,
    avg_attention2
)

print("Sample 1 — Meaningful Top 3:")
for token, score in top3_1:
    print(token, "→", round(score, 6))

print("\nSample 2 — Meaningful Top 3:")
for token, score in top3_2:
    print(token, "→", round(score, 6))

Sample 1 — Meaningful Top 3:
hair → 0.082552
scientists → 0.081474
unveil → 0.074925

Sample 2 — Meaningful Top 3:
nails → 0.100657
totally → 0.097479
equality → 0.073158


In [33]:
# Decode the tokenized inputs back into text

truncated_sentence1 = tokenizer.decode(
    inputs1["input_ids"][0],
    skip_special_tokens=True
)

truncated_sentence2 = tokenizer.decode(
    inputs2["input_ids"][0],
    skip_special_tokens=True
)

print("Sample 1 - Truncated sentence:")
print(truncated_sentence1)

print("\nSample 2 - Truncated sentence:")
print(truncated_sentence2)

Sample 1 - Truncated sentence:
thirtysomething scientists unveil doomsday clock of hair loss

Sample 2 - Truncated sentence:
dem rep. totally nails why congress is falling short on gender, racial equality


In [35]:
def get_layer_head_for_top3(attentions, tokens, top3):

    # Stack all layers
    attention_tensor = torch.stack(attentions)

    # Remove batch dimension
    # Shape: layers × heads × tokens × tokens
    attention_tensor = attention_tensor[:, 0, :, :, :]

    results = []

    for token, score in top3:

        # Find the token index
        token_indices = [
            i for i, t in enumerate(tokens)
            if t.replace("Ġ", "") == token
        ]

        if not token_indices:
            continue

        token_idx = token_indices[0]

        # Find highest attention across all layers and heads
        values = attention_tensor[:, :, :, token_idx]

        max_value = values.max()

        layer_idx, head_idx, source_idx = torch.where(
            values == max_value
        )[0][0], torch.where(values == max_value)[1][0], torch.where(values == max_value)[2][0]

        results.append({
            "token": token,
            "attention_score": score,
            "layer": layer_idx.item() + 1,
            "head": head_idx.item() + 1
        })

    return results

In [37]:
print([x for x in globals().keys() if "attention" in x.lower()])

['attentions1', 'attentions2', 'attention1', 'attention2', 'avg_attention1', 'avg_attention2']


In [38]:
import torch
import numpy as np

def get_layer_head(attentions, tokens, top3):

    # layers × batch × heads × tokens × tokens
    attention_tensor = torch.stack(attentions)

    # Remove batch dimension
    # layers × heads × tokens × tokens
    attention_tensor = attention_tensor[:, 0, :, :, :]

    results = []

    for token, score in top3:

        # Find the token position
        matching_indices = [
            i for i, t in enumerate(tokens)
            if t.replace("Ġ", "") == token
        ]

        if not matching_indices:
            print("Token not found:", token)
            continue

        token_idx = matching_indices[0]

        # Attention received by this token
        values = attention_tensor[:, :, :, token_idx]

        # Find the maximum raw attention
        flat_index = torch.argmax(values)

        layer, head, source_token = np.unravel_index(
            flat_index.item(),
            values.shape
        )

        results.append({
            "Token": token,
            "Attention Score": float(score),
            "Layer": layer + 1,
            "Head": head + 1
        })

    return results

In [39]:
layer_head_1 = get_layer_head(
    attentions1,
    tokens1,
    top3_1
)

print("SAMPLE 1")
for result in layer_head_1:
    print(result)

SAMPLE 1
{'Token': 'hair', 'Attention Score': 0.08255179971456528, 'Layer': np.int64(2), 'Head': np.int64(3)}
{'Token': 'scientists', 'Attention Score': 0.08147449046373367, 'Layer': np.int64(2), 'Head': np.int64(3)}
{'Token': 'unveil', 'Attention Score': 0.07492540776729584, 'Layer': np.int64(2), 'Head': np.int64(3)}


In [40]:
layer_head_2 = get_layer_head(
    attentions2,
    tokens2,
    top3_2
)

print("SAMPLE 2")
for result in layer_head_2:
    print(result)

SAMPLE 2
{'Token': 'nails', 'Attention Score': 0.10065698623657227, 'Layer': np.int64(2), 'Head': np.int64(3)}
{'Token': 'totally', 'Attention Score': 0.09747928380966187, 'Layer': np.int64(2), 'Head': np.int64(3)}
{'Token': 'equality', 'Attention Score': 0.07315784692764282, 'Layer': np.int64(2), 'Head': np.int64(3)}
